# 01 — Prompt template check and prompt lengths (step 1.3)

Builds real prompts with `src/prompts.py` and measures their length in tokens for every split.
These numbers set `max_model_len` for vLLM (Phase 1.5) and `max_seq_length` for training (Phase 3).

Runtime: CPU. BIRD train is unpacked from the HF dataset repo, which takes several minutes.

In [1]:
import os
REPO_DIR = "/content/text2sql-small-llm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sameerahmed-cloud/text2sql-small-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

Cloning into '/content/text2sql-small-llm'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 66 (delta 23), reused 44 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 28.23 KiB | 4.03 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/text2sql-small-llm


In [2]:
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # BIRD train is in a private HF repo

import json
from src.get_data import unpack_all
from src.data_prep import DRIVE_PROCESSED, db_path

paths = unpack_all(include_bird_train=True)

def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

splits = {n: read_jsonl(DRIVE_PROCESSED / f"{n}.jsonl") for n in ["train", "val", "test_bird", "test_spider"]}
print({n: len(r) for n, r in splits.items()})   # expect 14601, 1634, 1534, 1034

Mounted at /content/drive
Unpacking bird_dev (big datasets take several minutes)...
Unpacking spider (big datasets take several minutes)...


bird_train.zip: reconstructing file:   0%|          |  0.00B / 8.92GB            

bird_train.zip: downloading bytes:           |  0.00B            

Unpacking bird_train (big datasets take several minutes)...
✓ Paths saved to /content/data/raw/paths.json
{'train': 14601, 'val': 1634, 'test_bird': 1534, 'test_spider': 1034}


In [3]:
from transformers import AutoTokenizer
from src.prompts import build_prompt, measure_prompt_lengths

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct")

# One real Spider prompt, printed in full, as a visual check
r = splits["test_spider"][0]
print(build_prompt(db_path(r, paths), r["question"], r["evidence"], tok))
print("GOLD:", r["gold_sql"])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are an expert data analyst who writes correct SQLite queries. Use only the tables and columns that exist in the given database schema.<|im_end|>
<|im_start|>user
Database schema:
CREATE TABLE "stadium" (
"Stadium_ID" int,
"Location" text,
"Name" text,
"Capacity" int,
"Highest" int,
"Lowest" int,
"Average" int,
PRIMARY KEY ("Stadium_ID")
)
/* Sample values:
  Stadium_ID: 1, 2, 3
  Location: 'Raith Rovers', 'Ayr United', 'East Fife'
  Name: 'Stark''s Park', 'Somerset Park', 'Bayview Stadium'
  Capacity: 10104, 11998, 2000
  Highest: 4812, 2363, 1980
  Lowest: 1294, 1057, 533
  Average: 2106, 1477, 864
*/

CREATE TABLE "singer" (
"Singer_ID" int,
"Name" text,
"Country" text,
"Song_Name" text,
"Song_release_year" text,
"Age" int,
"Is_male" bool,
PRIMARY KEY ("Singer_ID")
)
/* Sample values:
  Singer_ID: 1, 2, 3
  Name: 'Joe Sharp', 'Timbaland', 'Justin Brown'
  Country: 'Netherlands', 'United States', 'France'
  Song_Name: 'You', 'Dangerous', 'Hey Oh'
  Song_release_

In [4]:
# Token lengths for every split (takes a few minutes: 18,803 prompts)
stats = [measure_prompt_lengths(recs, paths, tok, name) for name, recs in splits.items()]


train: 14601 prompts | mean 2276 | p50 1372 | p95 5611 | max 22230 tokens
  over  1024 tokens:  8707 (59.6%)
  over  2048 tokens:  4371 (29.9%)
  over  4096 tokens:  1191 (8.2%)
  longest databases: works_cycles (22230), baseball_1 (7784), mondial_geo (5679)

val: 1634 prompts | mean 2104 | p50 1267 | p95 7429 | max 7518 tokens
  over  1024 tokens:   901 (55.1%)
  over  2048 tokens:   534 (32.7%)
  over  4096 tokens:   207 (12.7%)
  longest databases: hockey (7518), sakila_1 (2980), retails (2702)

test_bird: 1534 prompts | mean 2713 | p50 3030 | p95 6182 | max 6296 tokens
  over  1024 tokens:  1325 (86.4%)
  over  2048 tokens:   828 (54.0%)
  over  4096 tokens:   320 (20.9%)
  longest databases: european_football_2 (6296), card_games (4337), california_schools (3557)

test_spider: 1034 prompts | mean 899 | p50 753 | p95 2084 | max 2105 tokens
  over  1024 tokens:   226 (21.9%)
  over  2048 tokens:    82 (7.9%)
  over  4096 tokens:     0 (0.0%)
  longest databases: dog_kennels (2105),